#### 01. mass-generator

- Description: 
  Script para geração de massa de dados fake.


-- David Costa

#### 02. Execution Configuration

In [0]:
%python

from pyspark.sql.functions import col, concat, lit, element_at, array, rand, round, when, current_timestamp

# Parâmetros para controle manual de ID.
START_ID = 1
NUM_RECORDS = 1000000 # Simulando 1M de registros.

# Caminho para salvar dados.
path = "/Volumes/lakeflow/a_raw/massive-stream-declarative-pipeline/"
path_customers = f"{path}customers/"
path_sales = f"{path}sales_events/"

#### 03. generate_data

###### 03.1 generate_data_customers

In [0]:
%python

def generate_data_customers(start_id):
    # Gerando dados de Clientes.
    customers_df = (
        spark.range(start_id, start_id + 50000)
        .select(
            col("id").alias("customer_id"),
            concat(lit("Customer_"), col("id")).alias("customer_name"),
            element_at(
                array(lit("Bronze"), lit("Silver"), lit("Gold"), lit("Platinum")),
                (rand() * 4 + 1).cast("int")
            ).alias("loyalty_tier")
        )
    )

    return customers_df

###### 03.2 generate_data_sales

In [0]:
%python

def generate_data_sales(start_id, num_records):
   
    # Gerando dados de Vendas.
    sales_df = spark.range(start_id, start_id + num_records).select(
        col("id").alias("sale_id"),
        (rand() * 50000 + start_id).cast("int").alias("customer_id"),
        element_at(array(lit("Physical"), lit("Virtual")), (rand() * 2 + 1).cast("int")).alias("store_type"),
        round(rand() * 50, 2).alias("shipping_fee"),
        round(rand() * 100, 2).alias("discount_amount"),
        when(rand() > 0.5, "SUMMER26").otherwise(None).alias("coupon_code"),
        when(col("store_type") == "Physical", (rand() * 100).cast("int")).otherwise(None).alias("seller_id"),
        element_at(array(lit("None"), lit("Standard"), lit("Extended")), (rand() * 3 + 1).cast("int")).alias("warranty_type"),
        round(rand() * 1000, 2).alias("amount"),
        current_timestamp().alias("timestamp")
    )
    
    return sales_df

#### 04. saving the data

In [0]:
%python

customers_df = generate_data_customers(START_ID)
sales_df = generate_data_sales(START_ID, NUM_RECORDS)

# Salvando como JSON/Parquet para simular Landing Zone.
sales_df.write.mode("append").json(path_sales)
customers_df.write.mode("overwrite").parquet(path_customers)